**Cleaning Data (Milestone 2) | Tyler Hilbert | April 14, 2026**

This is the cleaning data file from Milestone 1, but we have to add some things to help "future-proof" this file.

In the original file, I touched on some limitations of this file due to the "anonymization" of the original data. I am not going to factor those into this version of the file - that'll be something that gets added post-project before this gets moved into a live state. Instead, I am going to add some additional columns (Subject College & Mid Term/Final Grade Numerical equivalent) and tweak an existing column (College to Major College).

While I'm going through the file, I am going to do some little tweaks to the file - mostly removing some of the narrative pieces and moving them into the coding cells where appropriate (i.e. explaining what the code does). I'm going to add Markdown cells and highlight the header in yellow so the new piece(s) can be quickly identified.

**Importing Libraries and Looking at the data**

In [112]:
#Importing Libraries
import pandas as pd #importing pandas
import numpy as np #importing numpy

In [113]:
#Checking Out the Data Pt. 1 (Loading the data)
mtfull = pd.read_csv("mtgradesanon.csv") #Pulling in the mt grades file
mtfull.head() #running the head to make sure it works

,Unnamed: 0,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class
0,0,RW,AED,22860,KC,A-,B,ARCH,ID,202280,FR
1,1,RW,AED,22860,KC,A,A,ARCH,ID,202280,FR
2,2,RW,AED,22860,KC,A,A,ARCH,OTH,202280,JR
3,3,RW,AED,22860,KC,F,B,ARCH,ID,202280,SO
4,4,RW,AED,22860,KC,B,B,ARCH,ID,202280,FR


In [114]:
#Checking Out the Data Pt. 2 (Checking out the shape)
mtfull.shape #making sure the shape has come in successfully

(85341, 11)

In [115]:
#Checking Out the Data Pt. 3 (Checking out the Keys)
mtfull.keys() #checking out the keys

Index(['Unnamed: 0', 'Registration Status', 'Subject', 'Course', 'Campus',
       'Final Grade', 'Mid Term Grade', 'Department', 'Major',
       'Academic Period', 'Class'],
      dtype='object')

In [116]:
#Checking Out the Data Pt. 4 (Checking out the Types)
mtfull.dtypes #checking out the types of the dataframe

Unnamed: 0              int64
Registration Status    object
Subject                object
Course                  int64
Campus                 object
Final Grade            object
Mid Term Grade         object
Department             object
Major                  object
Academic Period         int64
Class                  object
dtype: object

**Replacing A Column Header**

After reviewing the shape and types of data in the dataframe, the first thing I'd like to do is remove Unnamed: 0 as a column header. This is an unfortunate consequence of removing the ID in the dummying file. I think the best way to handle this is transforming the name of the column to "Record ID." In hindsight, I could have used the logic for shuffling the course numbers with the Student IDs, but I wanted to minimize the risk of anything getting out. 

In [117]:
#Replacing A Column Header
#Unnamed: 0 is an unfortunate consequence of removing the ID from the last round. I opted to replace it with an artificial ID column header - this step will be removed in the "final" version
mtfull = mtfull.rename(columns={"Unnamed: 0":"Record ID"}) #renaming the column to Record ID - not exact, but looks nicer

**Making a Course Code Column**

The first manipulation we are doing in this file is creating a Course Code column. In its current state, the Subject and Course # are separate columns - in many reports, both are presented together. I think adding a Course Code column will allow for two options for future filters:
- Filtering by the full course code - this may be useful in combining with the department value
- Filtering by subject *and then* course number. This may be useful for people honing in on a specific subject before a course.

In [118]:
#Making a Course Code Column Pt. 1 (Converting Course to String)
mtfull["Course"] = mtfull["Course"].astype(str) #converting the course (int) to an object (str)
mtfull.dtypes #checking to make sure it worked

Record ID               int64
Registration Status    object
Subject                object
Course                 object
Campus                 object
Final Grade            object
Mid Term Grade         object
Department             object
Major                  object
Academic Period         int64
Class                  object
dtype: object

In [119]:
#Making a Course Code Column Pt. 2 (Creating the Course Code Column)
mtfull["Course Code"] = mtfull["Subject"] + " " + mtfull["Course"] #Formally combining them - the space ensures there is a space between the two
mtfull #it worked this time!

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code
0,0,RW,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860
1,1,RW,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860
2,2,RW,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860
3,3,RW,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860
4,4,RW,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860
...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,RW,MDJ,20288,KC,B+,B,MDJ,FM,202310,SO,MDJ 20288
85337,85337,RW,MDJ,20288,KC,A,A,MDJ,FM,202310,JR,MDJ 20288
85338,85338,DD,MDJ,20288,KC,NaN,NaN,MDJ,COMM,202310,JR,MDJ 20288
85339,85339,RW,MDJ,20288,KC,B+,B+,MDJ,FM,202310,SO,MDJ 20288


**Translating Status Codes**

The next step is moving to registration status codes. There are a lot of different codes in the system that mean the same thing, just impacted by timing/who does the thing. Writing them out would help explain to someone why certain grade combinations happen (i.e. why someone who dropped a course still has a final and midterm grade when they really shouldn't)

In [120]:
#Translating the Reg Status Codes Pt. 1 (Checking the Codes)
#There are a lot of codes in the system that mean the same thing, but with small differences (i.e. DD and W8 both mean drop, just when the drop happened)
regstatuscodes = mtfull["Registration Status"].unique() #Making a list of all the registration status codes in the file so I know what to change out
regstatuscodes #Generates a list of 15 codes that are present in the file

array(['RW', 'RE', 'WW', 'DD', 'W8', 'ND', 'R2', 'SF', 'B1', 'WD', 'NF',
       'RA', 'AW', 'DR', 'B5'], dtype=object)

In [121]:
#Translating the Reg Status Codes Pt. 2 (Making the Map)
regstatuscodestranslated = ["Registered","Std Withdrawn","Stopped Attending - Failed", "Admin Dropped","Std Dropped","Admin Dropped","Never Attended - Failed", "Registered", "Std Withdrawn", "Registered", "Admin Withdrawn", "Audited", "Admin Dropped", "Admin Withdrawn", "Std Dropped"]
regstatuscodestranslated

['Registered',
 'Std Withdrawn',
 'Stopped Attending - Failed',
 'Admin Dropped',
 'Std Dropped',
 'Admin Dropped',
 'Never Attended - Failed',
 'Registered',
 'Std Withdrawn',
 'Registered',
 'Admin Withdrawn',
 'Audited',
 'Admin Dropped',
 'Admin Withdrawn',
 'Std Dropped']

In [122]:
#Translating the Reg Status Codes Pt. 3 (Matching Codes to Full)
mapregstatuscode = dict(zip(regstatuscodes, regstatuscodestranslated)) #Making a dictionary that matches the codes to the full version
mapregstatuscode #running it so it works

{'RW': 'Registered',
 'RE': 'Std Withdrawn',
 'WW': 'Stopped Attending - Failed',
 'DD': 'Admin Dropped',
 'W8': 'Std Dropped',
 'ND': 'Admin Dropped',
 'R2': 'Never Attended - Failed',
 'SF': 'Registered',
 'B1': 'Std Withdrawn',
 'WD': 'Registered',
 'NF': 'Admin Withdrawn',
 'RA': 'Audited',
 'AW': 'Admin Dropped',
 'DR': 'Admin Withdrawn',
 'B5': 'Std Dropped'}

In [123]:
#Translating the Reg Status Codes Pt. 4 (Replacing the Codes w/ Full)
mtfull["Registration Status"] = mtfull["Registration Status"].map(mapregstatuscode) #using the .map(), this is creating a new version of the dataframe with the full code.
mtfull #running to make sure it worked

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860
...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,202310,SO,MDJ 20288
85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,202310,JR,MDJ 20288
85338,85338,Admin Dropped,MDJ,20288,KC,NaN,NaN,MDJ,COMM,202310,JR,MDJ 20288
85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,202310,SO,MDJ 20288


<mark>**Making a Major College Column**</mark>

**NEW** - We are going to tweak the name of this column to be Major College - this will differentiate from the Subject College made in the future.

In [124]:
#Making a Major College Column Pt. 1 (Looking at Values)
mtfull["Major"].unique() #Checking out the values that are in the Major column

array(['ID', 'OTH', 'ARCH', 'FM', 'COMM', 'ARTH', 'FD', 'DNST', 'TDTP',
       'SART', 'COMA', 'DMP', 'VCD', 'JNL', 'EMAT', 'ADV', 'THEA', 'ARCS',
       'PR', 'ARTE', 'MUST', 'PHOT', 'MUS', 'MUED', 'MUT', 'APMD', 'UXDE',
       'DANC'], dtype=object)

In [125]:
#Making a Major College Column Pt. 2 (Making College Variables)
caedmajor = ["ID", "ARCH", "COMA", "ARCS"] #Majors in CAED
ccimajor = ["COMM", "DMP", "VCD", "JNL", "EMAT", "ADV", "PR", "PHOT", "APMD","UXDE"] #Majors in CCI
cotamajor = ["FM", "ARTH", "FD", "DNST", "TDTP", "SART", "THEA", "ARTE", "MUST", "MUS", "MUED", "MUT","DANC"] #Majors in CotA
othermajor = ["OTH"] #Majors outside those colleges

In [126]:
#Making a Major College Column Pt. 3 (Making Condition and Outcomes)
majcond = [ #So did some looking, and isin seems to be the way forward. Before was trying to find exact matches, this way is saying "Hey - check this list and see if it is there first"
    mtfull["Major"].isin(caedmajor),
    mtfull["Major"].isin(ccimajor),
    mtfull["Major"].isin(cotamajor),
    mtfull["Major"].isin(othermajor)] 

majout = ["CAED", "CCI", "CotA", "Other"]

In [127]:
#Making a Major College Column Pt. 4 (Making the Column)
mtfull["Major College"] = np.select(majcond,majout, "Missing") #After thinking about it, I realized that it may be better to have a missing value - that way if I somehow missed something I can catch it
mtfull

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860,Other
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED
...,...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,202310,SO,MDJ 20288,CotA
85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,202310,JR,MDJ 20288,CotA
85338,85338,Admin Dropped,MDJ,20288,KC,NaN,NaN,MDJ,COMM,202310,JR,MDJ 20288,CCI
85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,202310,SO,MDJ 20288,CotA


In [128]:
#Making a Major College Column Pt. 5 (Checking for missing values)
collegemajcheck = mtfull.groupby("Major College").count()["Record ID"]
collegemajcheck

Major College
CAED     11366
CCI      15236
CotA     32185
Other    26554
Name: Record ID, dtype: int64

<mark>**Making a Subject College Column**</mark>

So we are going to replicate the steps above, but instead of making a column based on major, we are going to make it based on the course subject. This will help with filtering by the subject easier since we can look at all the programs in a given unit as opposed to a college. The code is going to replicate much of the above code, but without the Other option since all the courses will be offered in one of the three colleges. However, we will use "Missing" again to try and find any subjects I missed in the coding process.

In [129]:
#Making a Subject College Column Pt. 1 (Checking out the Subject Codes)
mtfull["Subject"].unique()

array(['AED', 'ARCH', 'ARCS', 'ART', 'ARTH', 'ARTS', 'CCI', 'CMGT',
       'COMM', 'DAN', 'EMAT', 'FDM', 'ID', 'MDJ', 'MUS', 'THEA', 'VCD'],
      dtype=object)

In [130]:
#Making a Subject College Column Pt. 2 (Making College Variables)
caedsubject = ["AED", "ARCH", "ARCS", "CMGT", "ID"] #Subjects in CAED
ccisubject = ["CCI", "COMM", "EMAT", "MDJ", "VCD"] #Subjects in CCI
cotasubject = ["ART", "ARTH", "ARTS", "DAN", "FDM", "MUS", "THEA"] #Subjects in CotA

In [131]:
#Making a Subject College Column Pt. 3 (Making Condition and Outcomes)
subcond = [
    mtfull["Subject"].isin(caedsubject),
    mtfull["Subject"].isin(ccisubject),
    mtfull["Subject"].isin(cotasubject)] 

subout = ["CAED", "CCI", "CotA"]

In [132]:
#Making a Subject College Column Pt. 4 (Making the Column)
mtfull["Subject College"] = np.select(subcond,subout, "Missing") #This is where a missing value will come in handy - it'll help identify subjects I may have missed
mtfull

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860,Other,CAED
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED,CAED
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,202310,SO,MDJ 20288,CotA,CCI
85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,202310,JR,MDJ 20288,CotA,CCI
85338,85338,Admin Dropped,MDJ,20288,KC,NaN,NaN,MDJ,COMM,202310,JR,MDJ 20288,CCI,CCI
85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,202310,SO,MDJ 20288,CotA,CCI


In [133]:
collegesubcheck = mtfull.groupby("Subject College").count()["Record ID"]
collegesubcheck

Subject College
CAED    15109
CCI     27057
CotA    43175
Name: Record ID, dtype: int64

**Writing out the terms**

In [134]:
#Making a Term Column Pt. 1 (Identifying periods)
shortterm = mtfull["Academic Period"].unique()
shortterm

array([202280, 202310, 202380, 202410, 202480, 202510, 202580, 202610])

In [135]:
#Making a Term Column Pt. 2 (Defining the periods)
fullterm = ["Fall 2022", "Spring 2023", "Fall 2023", "Spring 2024", "Fall 2024", "Spring 2025", "Fall 2025", "Spring 2026"]
fullterm

['Fall 2022',
 'Spring 2023',
 'Fall 2023',
 'Spring 2024',
 'Fall 2024',
 'Spring 2025',
 'Fall 2025',
 'Spring 2026']

In [136]:
#Making a Term Column Pt. 3 (Making the term map)
mapterm = dict(zip(shortterm, fullterm))
mapterm

{np.int64(202280): 'Fall 2022',
 np.int64(202310): 'Spring 2023',
 np.int64(202380): 'Fall 2023',
 np.int64(202410): 'Spring 2024',
 np.int64(202480): 'Fall 2024',
 np.int64(202510): 'Spring 2025',
 np.int64(202580): 'Fall 2025',
 np.int64(202610): 'Spring 2026'}

In [137]:
#Making a Term Column Pt. 4 (Overwriting the Academic Period column)
mtfull["Academic Period"] = mtfull["Academic Period"].map(mapterm)
mtfull

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,Fall 2022,JR,AED 22860,Other,CAED
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,Fall 2022,SO,AED 22860,CAED,CAED
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI
85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,Spring 2023,JR,MDJ 20288,CotA,CCI
85338,85338,Admin Dropped,MDJ,20288,KC,NaN,NaN,MDJ,COMM,Spring 2023,JR,MDJ 20288,CCI,CCI
85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI


**Withdrawals and Mid Term Grades**

In [138]:
#Withdrew W/o MT Grade Entered Pt. 1 (Making a new column)
mtfull["Withdrew No MT"] = np.where((mtfull["Final Grade"] == "W") & (mtfull["Mid Term Grade"].isna()), True, False) #It worked this time - huzzah
mtfull

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Withdrew No MT
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,Fall 2022,JR,AED 22860,Other,CAED,False
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,Fall 2022,SO,AED 22860,CAED,CAED,False
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI,False
85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,Spring 2023,JR,MDJ 20288,CotA,CCI,False
85338,85338,Admin Dropped,MDJ,20288,KC,NaN,NaN,MDJ,COMM,Spring 2023,JR,MDJ 20288,CCI,CCI,False
85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI,False


In [139]:
#Withdrew W/o MT Grade Entered Pt. 2 (Counting the Instances)
mtfull.groupby("Withdrew No MT").count()["Record ID"]

Withdrew No MT
False    83963
True      1378
Name: Record ID, dtype: int64

In [140]:
#Withdrew W/o MT Grade Entered Pt. 3 (Adding the Ws)
mtfull["Mid Term Grade"] = np.where(mtfull["Withdrew No MT"] == True, "W", mtfull["Mid Term Grade"])
mtfull["Mid Term Grade"].unique()

array(['B', 'A', nan, 'A-', 'C', 'C+', 'D', 'B+', 'B-', 'C-', 'F', 'S',
       'D+', 'W', 'U', 'SF', 'NF'], dtype=object)

**Identifying Dropped Grades**

In [141]:
#Adding a DR Grade Pt. 1 (Writing the conditional column)
mtfull["Dropped no MT"] = np.where(((mtfull["Registration Status"] == "Std Dropped") | (mtfull["Registration Status"] == "Admin Dropped")) & (mtfull["Mid Term Grade"].isna()), True, False)

In [142]:
#Adding a DR Grade Pt. 2 (Counting how many instances)
mtfull.groupby("Dropped no MT").count()["Record ID"]

Dropped no MT
False    83989
True      1352
Name: Record ID, dtype: int64

In [143]:
#Adding a DR Grade Pt. 3 (Plugging in the DR values)
mtfull["Mid Term Grade"] = np.where(mtfull["Dropped no MT"] == True, "DR", mtfull["Mid Term Grade"])
mtfull["Mid Term Grade"].unique()

array(['B', 'A', nan, 'A-', 'C', 'C+', 'D', 'B+', 'B-', 'C-', 'F', 'S',
       'D+', 'DR', 'W', 'U', 'SF', 'NF'], dtype=object)

**Removing X Grades**

In [144]:
#Removing X grades Pt. 1 (Checking for the X grade presence)
mtfull["Final Grade"].unique()

array(['A-', 'A', 'F', 'B', nan, 'D+', 'C', 'B-', 'B+', 'C-', 'W', 'C+',
       'S', 'D', 'XD', 'XF', 'SF', 'NF', 'AU', 'XSF', 'XC-', 'IN', 'XD+',
       'NR', 'XNF', 'U'], dtype=object)

In [145]:
#Removing X grades Pt. 2 (Removing and Checking for the grade)
mtfull["Final Grade"] = mtfull["Final Grade"].str.lstrip("X")
mtfull["Mid Term Grade"].unique()

array(['B', 'A', nan, 'A-', 'C', 'C+', 'D', 'B+', 'B-', 'C-', 'F', 'S',
       'D+', 'DR', 'W', 'U', 'SF', 'NF'], dtype=object)

**Adding a DR to Final Grades**

In [146]:
#Adding DR to final grades Pt. 1 (Making the conditional column)
mtfull["Dropped no Fin"] = np.where(((mtfull["Registration Status"] == "Std Dropped") | (mtfull["Registration Status"] == "Admin Dropped")) & (mtfull["Final Grade"].isna()), True, False)
mtfull.groupby("Dropped no Fin").count()["Record ID"]

Dropped no Fin
False    83961
True      1380
Name: Record ID, dtype: int64

In [147]:
#Adding DR to final grades Pt. 2 (Adding the DR Grade)
mtfull["Final Grade"] = np.where(mtfull["Dropped no Fin"] == True, "DR", mtfull["Final Grade"])
mtfull["Final Grade"].unique()

array(['A-', 'A', 'F', 'B', nan, 'D+', 'C', 'B-', 'B+', 'C-', 'W', 'C+',
       'S', 'D', 'DR', 'SF', 'NF', 'AU', 'IN', 'NR', 'U'], dtype=object)

**Discrepancies between MT and Final Grades**

Something that set alarms off in my head was the difference in "True" values between the Final Grades DR and the Mid Term Grades DR values. After thinking about it, I realized it may be due to non-student drops, which can occur *after* the mid term grades are posted. Again, students can request late drops or expungements from their records, which could result in wonky grades. I want to confirm this, so I am going to look at records that have a Final Grade of DR and a Mid Term grade that isn't a DR.

In [148]:
#Checking disc between MT and final grades Pt. 1 (Looking at the instances)
mtfull.loc[(mtfull["Final Grade"] == "DR") & (mtfull["Mid Term Grade"] != "DR")]

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Withdrew No MT,Dropped no MT,Dropped no Fin
2384,2384,Admin Dropped,ARCH,16841,KC,DR,SF,ARCH,FM,Fall 2024,FR,ARCH 16841,CotA,CAED,False,False,True
5169,5169,Admin Dropped,ARCH,11467,KC,DR,NF,ARCH,ARCH,Fall 2023,FR,ARCH 11467,CAED,CAED,False,False,True
6249,6249,Admin Dropped,ARCH,15217,KC,DR,NF,ARCH,OTH,Spring 2025,FR,ARCH 15217,Other,CAED,False,False,True
15281,15281,Admin Dropped,ARTH,17400,KC,DR,NF,ART,FD,Fall 2024,SR,ARTH 17400,CotA,CotA,False,False,True
21320,21320,Admin Dropped,CMGT,21745,KC,DR,B+,ARCH,ARCH,Spring 2023,JR,CMGT 21745,CAED,CAED,False,False,True
21346,21346,Admin Dropped,CMGT,21745,KC,DR,NF,ARCH,COMA,Spring 2024,SO,CMGT 21745,CAED,CAED,False,False,True
21505,21505,Admin Dropped,CMGT,21745,KC,DR,B,ARCH,ARCH,Spring 2025,JR,CMGT 21745,CAED,CAED,False,False,True
23375,23375,Admin Dropped,COMM,17591,KC,DR,NF,COMM,OTH,Spring 2023,FR,COMM 17591,Other,CCI,False,False,True
25210,25210,Admin Dropped,COMM,17591,KC,DR,NF,COMM,ADV,Spring 2024,FR,COMM 17591,CCI,CCI,False,False,True
25910,25910,Admin Dropped,COMM,17591,KC,DR,NF,COMM,VCD,Fall 2024,SR,COMM 17591,CCI,CCI,False,False,True


Looking at the list above, my read was right - the discrepancies were due to admin drops. Students most often get this when there are extenuating circumstances (i.e. hospitalizations) where they couldn't finish in time or didn't realize they were registered for classes and dropped on time. Since a midterm grade was still assigned, it would still reflect in the system. However, I want to makes sure I am catching the difference between the two, so I'm going to run a count.

In [149]:
#Checking disc between MT and final grades Pt. 2 (Checking the count)
mtfull.loc[(mtfull["Final Grade"] == "DR") & (mtfull["Mid Term Grade"] != "DR")].count()

Record ID              28
Registration Status    28
Subject                28
Course                 28
Campus                 28
Final Grade            28
Mid Term Grade         28
Department             28
Major                  28
Academic Period        28
Class                  28
Course Code            28
Major College          28
Subject College        28
Withdrew No MT         28
Dropped no MT          28
Dropped no Fin         28
dtype: int64

<mark>**Converting MT & Final Grades to Numerical Values**</mark>

While brainstorming ideas in class, I was trying to think of the best way to convert the data to a heat map. The idea came up to convert the grades into numbers, which I think is a great idea. I then had to brainstorm what numbers to convert them to that is readable. After some research, I think the best conversion rate will be the university's GPA scale (https://catalog.kent.edu/academic-policies/grade-point-average/). This is fitting since these files will be read by those at Kent State, and the grade scale is common knowledge.

I'm going to first pull all of the unique MT and Final Grades to see what values exist, and then I'm going to write out the conversion chart for reference.

In [150]:
#Converting Grades to Numerical Values Pt. 1 (Checking Unique Final Grades)
mtfull["Final Grade"].unique()

array(['A-', 'A', 'F', 'B', nan, 'D+', 'C', 'B-', 'B+', 'C-', 'W', 'C+',
       'S', 'D', 'DR', 'SF', 'NF', 'AU', 'IN', 'NR', 'U'], dtype=object)

In [151]:
#Converting Grades to Numerical Values Pt. 2 (Checking Unique MT Grades)
mtfull["Mid Term Grade"].unique()

array(['B', 'A', nan, 'A-', 'C', 'C+', 'D', 'B+', 'B-', 'C-', 'F', 'S',
       'D+', 'DR', 'W', 'U', 'SF', 'NF'], dtype=object)

OK - so now that we have the list of all grades, I am going to make my conversion chart in the below cell for reference later. 

Something I realized will be tricky is the conversion rates for courses that do not contribute to GPAs. Attempting to logic it out, I think the best way to handle those values is as follows:
- DR: will be a blank value. since they never earned a grade, there is no point in assigning a value since they would lead to a higher concentration of values in a certain spot (you can't earn a higher/lower grade after, so it'll show no movement in grades and throw off the visual)
- NR: will be an NaN. NR grades are assigned when the instructor never reported a grade. this is not count 
- AU: will be a blank value - these are people auditing the course, so they never get a grade.
- S: will be a 4. S grades do not contribute to GPA, but they are a passing grade. Presumably there would be movement from S to U and vice versa, so U will be assigned a 0.
- U: will be a 0. See the logic for S
- W: will be a 0. This is probably the most controversial one. W's do not contribute to GPA, but it is still a "negative" outcome for the course and is calculated in other retention efforts as a negative outcome (i.e. DFW rates are D, F and W grades). Another thought I had was a -1 to further differentiate, but I think that would be too much of an issue and impact a heat map (it'd add another column and row, so that could make it look weird/throw stuff off). I may toy with bringing back the W's as -1's later when I get to the heat maps and see what it looks like, but that's a problem for future me!
- IN: will be a blank value. IN grades are put whenever students get an extension on entering their grades. It is interpreted similarly to W's (doesn't impact GPA), but since there is a chance for this to change, I don't want to count it (students could complete the work and come back with a passing grade)

I will also need to consider how to convert NaN values in the MT column (i.e. people who never got a MT grade because it wasn't entered/never got one). I'm thinking that, whenever I make the heat map, these will be removed and some form of indicator will be kept indicating how many values were lost due to non-entry. I think this is an acceptable approach since it promotes transparency of why they are excluded, and will encourage leadership to remind faculty to enter MT grades so we can have as accurate data points as possible. This may also be where I enter other NaN values - that way leadership knows how many people got those grade types in the course.

I'm going to use the mapping method from class here - I thought about replicating the method I used earlier (creating different variables with lists), but decided this approach would be good to use for practice.

**Grade Conversion Chart**
- A = 4.0
- A- = 3.7
- B+ = 3.3
- B = 3.0
- B- = 2.7
- C+ = 2.3
- C = 2.0
- C- = 1.7
- D+ = 1.3
- D = 1.0
- F = 0.0
- W = 0.0
- S = 4.0
- U = 0.0
- SF = 0.0
- NF =  0.0
- AU = NaN
- NaN = NaN
- DR =  NaN
- IN =  NaN
- NR = NaN

In [152]:
#Converting Grades to Numerical Values Pt. 3 (Making the Number Map)
gpamap = {"A":4.0, "A-":3.7, "B+":3.3, "B":3.0, "B-":2.7, "C+":2.3, "C":2.0, "C-":1.7, "D+":1.3,
          "D":1.0, "F":0.0, "W":0.0, "S":4.0, "U":0.0, "SF":0.0, "NF":0.0,
          "AU": "NaN", "DR":"NaN", "IN":"NaN", "NR":"NaN", "NaN":"NaN"}

In [153]:
#Converting Grades to Numerical Values Pt. 3 (Adding the Final Grade Number Column)
mtfull["Final Grade Number"] = mtfull["Final Grade"].map(gpamap)
mtfull

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Withdrew No MT,Dropped no MT,Dropped no Fin,Final Grade Number
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,3.7
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,4.0
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,Fall 2022,JR,AED 22860,Other,CAED,False,False,False,4.0
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,Fall 2022,SO,AED 22860,CAED,CAED,False,False,False,0.0
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI,False,False,False,3.3
85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,Spring 2023,JR,MDJ 20288,CotA,CCI,False,False,False,4.0
85338,85338,Admin Dropped,MDJ,20288,KC,DR,DR,MDJ,COMM,Spring 2023,JR,MDJ 20288,CCI,CCI,False,True,True,NaN
85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI,False,False,False,3.3


In [154]:
#Converting Grades to Numerical Values Pt. 4 (Adding the Mid Term Grade Number Column)
mtfull["Mid Term Grade Number"] = mtfull["Mid Term Grade"].map(gpamap)
mtfull

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Withdrew No MT,Dropped no MT,Dropped no Fin,Final Grade Number,Mid Term Grade Number
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,3.7,3.0
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,4.0,4.0
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,Fall 2022,JR,AED 22860,Other,CAED,False,False,False,4.0,4.0
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,Fall 2022,SO,AED 22860,CAED,CAED,False,False,False,0.0,3.0
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,3.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI,False,False,False,3.3,3.0
85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,Spring 2023,JR,MDJ 20288,CotA,CCI,False,False,False,4.0,4.0
85338,85338,Admin Dropped,MDJ,20288,KC,DR,DR,MDJ,COMM,Spring 2023,JR,MDJ 20288,CCI,CCI,False,True,True,NaN,NaN
85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI,False,False,False,3.3,3.3


It was at this point that I realized these may not be saving as numbers and could not be used as continuous data later. So I opted to check the data types

In [155]:
#Converting Grades to Numerical Values Pt. 5 (Checking the data types)
mtfull.dtypes

Record ID                 int64
Registration Status      object
Subject                  object
Course                   object
Campus                   object
Final Grade              object
Mid Term Grade           object
Department               object
Major                    object
Academic Period          object
Class                    object
Course Code              object
Major College            object
Subject College          object
Withdrew No MT             bool
Dropped no MT              bool
Dropped no Fin             bool
Final Grade Number       object
Mid Term Grade Number    object
dtype: object

Yep - it's an object. So I'll want to convert it to a numerical value. My first thought was to try an integer, so I used the formula below

In [156]:
#Converting Grades to Numerical Values Pt. 6 (Converting to int - Oops)
#mtfull["Final Grade Number"] = mtfull["Final Grade Number"].astype(int)
#mtfull["Mid Term Grade Number"] = mtfull["Mid Term Grade Number"].astype(int)
#mtfull.dtypes

So that didn't work. It is due to the NaN values we've got hanging out that cannot be converted. I did some looking and found information about the pd.to_numeric function (https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html). This lets me keep the blank values in my data 

In [157]:
#Converting Grades to Numerical Values Pt. 7 (Converting to Numerical Data)
mtfull["Final Grade Number"] = pd.to_numeric(mtfull["Final Grade Number"], errors = "coerce") #Restating that we want the data to become numeric and keep the blank values.
mtfull["Mid Term Grade Number"] = pd.to_numeric(mtfull["Mid Term Grade Number"], errors = "coerce")
mtfull.dtypes

Record ID                  int64
Registration Status       object
Subject                   object
Course                    object
Campus                    object
Final Grade               object
Mid Term Grade            object
Department                object
Major                     object
Academic Period           object
Class                     object
Course Code               object
Major College             object
Subject College           object
Withdrew No MT              bool
Dropped no MT               bool
Dropped no Fin              bool
Final Grade Number       float64
Mid Term Grade Number    float64
dtype: object

<mark>**Making the full file**</mark>

I realized I never gave this part its own section - oops! I want to indicate where the full file is made so I have it for reference in the future.

In [158]:
mtfull.to_csv("mtfullclean.csv")

**Removing the OTH Values**

OK - now that we have done all the manipulation needed, the mtfull variable can be used to help analyze the grades earned by *all* students in the classes above. Now we need to move into the intervention space, which would only impact students in the affected major(s). As such, we will make a variable that removes the OTH majors.

In [159]:
#Removing OTH Majors
mthubonly = mtfull[mtfull["Major"] != "OTH"].copy()
mthubonly

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Withdrew No MT,Dropped no MT,Dropped no Fin,Final Grade Number,Mid Term Grade Number
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,3.7,3.0
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,4.0,4.0
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,Fall 2022,SO,AED 22860,CAED,CAED,False,False,False,0.0,3.0
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,3.0,3.0
5,5,Registered,AED,22860,KC,B,NaN,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,3.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI,False,False,False,3.3,3.0
85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,Spring 2023,JR,MDJ 20288,CotA,CCI,False,False,False,4.0,4.0
85338,85338,Admin Dropped,MDJ,20288,KC,DR,DR,MDJ,COMM,Spring 2023,JR,MDJ 20288,CCI,CCI,False,True,True,NaN,NaN
85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI,False,False,False,3.3,3.3


**Making the Intervention Column**

OK - now we get to the most complex part of the whole script. We need to make the different conditions for students to qualify for interventions from advisors. These conditions are set by the college, so I am going to use their conditions:
- Before the current term, FR/SO students had to get a C or below to qualify for an intervention, while JR/SR needed an F to qualify.
- Now, FR/SO students need a D+ or below to qualify. JR/SR still only need an F.
- We also need to consider dropped students, withdrawals, and NF grades - all of those students would not qualify since they are no longer registered for the course. SF grades still count, since the student attended at least some of the course and, if they return, maybe could finish the course and get a passing grade.
- We also need to differentiate between FR/SO and JR/SR students, so we need a condition for them too.
- We also also have to define the pre-Spring 2026 terms and the post-Spring 2026 terms.

With all that said - I found the best way to proceed is to define the variables first, then the conditions. There are a lot of variables for each category, followed by specific conditions. The FR/SO conditions are the most dense, followed by the JR/SR, and then the ones just checking their status.

In [160]:
#Making the Intervention Column Pt. 1 (Quick Reference for Grades)
mthubonly["Mid Term Grade"].unique()

array(['B', 'A', nan, 'A-', 'C', 'C+', 'D', 'B+', 'B-', 'C-', 'F', 'S',
       'D+', 'DR', 'W', 'SF', 'NF', 'U'], dtype=object)

In [161]:
#Making the Intervention Column Pt. 2 (Making all the variables)
pre202610frsopassgrade = ["A", "A-", "B+","B","B-","C+","S"]
pre202610frsofailgrade = ["C", "C-", "D+", "D", "F","U","SF"]
post202610frsopassgrade = ["A", "A-", "B+","B","B-","C+","C", "C-","S"]
post202610frsofailgrade = ["D+", "D", "F","U", "SF"]
jrsrpassgrade = ["A", "A-", "B+","B","B-","C+","C","C-","D+","D","S"]
jrsrfailgrade = ["F","U","SF"]
dropgrade = ["DR"]
wgrade = ["W"]
nfgrade = ["NF"]
frsocheck = ["FR","SO"]
jrsrcheck = ["JR","SR"]
pre202610terms = ["Fall 2022", "Spring 2023", "Fall 2023", "Spring 2024", "Fall 2024", "Spring 2025", "Fall 2025"]
post202610terms = ["Spring 2026"]

In [162]:
#Making the Intervention Column Pt. 3 (Making all the conditions and outcomes)
intercond = [
    ((mthubonly["Mid Term Grade"].isin(pre202610frsopassgrade)) & (mthubonly["Class"].isin(frsocheck)) & (mthubonly["Academic Period"].isin(pre202610terms))),
    ((mthubonly["Mid Term Grade"].isin(pre202610frsofailgrade)) & (mthubonly["Class"].isin(frsocheck)) & (mthubonly["Academic Period"].isin(pre202610terms))),
    ((mthubonly["Mid Term Grade"].isin(post202610frsopassgrade)) & (mthubonly["Class"].isin(frsocheck))  & (mthubonly["Academic Period"].isin(post202610terms))),
    ((mthubonly["Mid Term Grade"].isin(post202610frsofailgrade)) & (mthubonly["Class"].isin(frsocheck))  & (mthubonly["Academic Period"].isin(post202610terms))),
    ((mthubonly["Mid Term Grade"].isin(jrsrpassgrade)) & (mthubonly["Class"].isin(jrsrcheck))),
    ((mthubonly["Mid Term Grade"].isin(jrsrfailgrade)) & (mthubonly["Class"].isin(jrsrcheck))),
    mthubonly["Mid Term Grade"].isin(dropgrade),
    mthubonly["Mid Term Grade"].isin(wgrade),
    mthubonly["Mid Term Grade"].isin(nfgrade)
]

interout = ["Passing MT Grade", "FR/SO Intervention Needed", "Passing MT Grade", "FR/SO Intervention Needed", "Passing MT Grade", "JR/SR Intervention Needed", "Dropped Course", "Withdrew From Course", "Never Attended Course"]

In [163]:
#Making the Intervention Column Pt. 4 (Creating the Intervention column)
mthubonly["MT Grade Status"] = np.select(intercond,interout, "No MT Grade Entered") #Remembering that the last bit is the "Catch all" value, I chose to make it the No MT Grade Entered
mthubonly.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Withdrew No MT,Dropped no MT,Dropped no Fin,Final Grade Number,Mid Term Grade Number,MT Grade Status
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,3.7,3.0,Passing MT Grade
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,4.0,4.0,Passing MT Grade
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,Fall 2022,SO,AED 22860,CAED,CAED,False,False,False,0.0,3.0,Passing MT Grade
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,3.0,3.0,Passing MT Grade
5,5,Registered,AED,22860,KC,B,NaN,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,3.0,NaN,No MT Grade Entered


In [164]:
#Making the Intervention Column Pt. 5 (Checking the Values)
mthubonly.groupby("MT Grade Status").count()["Record ID"]

MT Grade Status
Dropped Course                 927
FR/SO Intervention Needed     5460
JR/SR Intervention Needed      652
Never Attended Course           82
No MT Grade Entered           5697
Passing MT Grade             45021
Withdrew From Course           948
Name: Record ID, dtype: int64

In order to confirm the No MT Grade values are accurate, I ran another report that checked if the values are true. When I did, I found that there were five values that were not NA. This tells me two things:
1. I missed something in my conditions earlier.
2. I need to see what those values are to see if they are true or not.

In [165]:
#Making the Intervention Column Pt. 5 (Checking the No MT Grade Values)
mthubonly.loc[(mthubonly["MT Grade Status"] == "No MT Grade Entered") & (mthubonly["Mid Term Grade"].isna())]

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Withdrew No MT,Dropped no MT,Dropped no Fin,Final Grade Number,Mid Term Grade Number,MT Grade Status
5,5,Registered,AED,22860,KC,B,NaN,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,3.0,NaN,No MT Grade Entered
40,40,Registered,AED,22860,KC,B,NaN,ARCH,ID,Fall 2022,JR,AED 22860,CAED,CAED,False,False,False,3.0,NaN,No MT Grade Entered
45,45,Registered,AED,22860,KC,B-,NaN,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,2.7,NaN,No MT Grade Entered
46,46,Registered,AED,22860,KC,A,NaN,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,4.0,NaN,No MT Grade Entered
50,50,Registered,AED,22860,KC,A,NaN,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,4.0,NaN,No MT Grade Entered
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85294,85294,Registered,MDJ,20288,KC,C+,NaN,MDJ,FM,Spring 2023,SR,MDJ 20288,CotA,CCI,False,False,False,2.3,NaN,No MT Grade Entered
85304,85304,Std Withdrawn,MDJ,20288,KC,A,NaN,MDJ,COMM,Spring 2023,SO,MDJ 20288,CCI,CCI,False,False,False,4.0,NaN,No MT Grade Entered
85319,85319,Registered,MDJ,20288,KC,A,NaN,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI,False,False,False,4.0,NaN,No MT Grade Entered
85334,85334,Registered,MDJ,20288,KC,A,NaN,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI,False,False,False,4.0,NaN,No MT Grade Entered


In [166]:
#Making the Intervention Column Pt. 6 (Checking the No MT Grades w/ to see what is going on)
mthubonly.loc[(mthubonly["MT Grade Status"] == "No MT Grade Entered") & (mthubonly["Mid Term Grade"].notna())]

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Withdrew No MT,Dropped no MT,Dropped no Fin,Final Grade Number,Mid Term Grade Number,MT Grade Status
19608,19608,Registered,CMGT,16841,KC,D,A-,ARCH,COMA,Fall 2022,GR,CMGT 16841,CAED,CAED,False,False,False,1.0,3.7,No MT Grade Entered
20786,20786,Registered,CMGT,17964,KC,NaN,A,ARCH,COMA,Spring 2026,GR,CMGT 17964,CAED,CAED,False,False,False,NaN,4.0,No MT Grade Entered
20789,20789,Registered,CMGT,17964,KC,NaN,B+,ARCH,COMA,Spring 2026,GR,CMGT 17964,CAED,CAED,False,False,False,NaN,3.3,No MT Grade Entered
20879,20879,Registered,CMGT,19400,KC,A-,C+,ARCH,COMA,Spring 2023,GR,CMGT 19400,CAED,CAED,False,False,False,3.7,2.3,No MT Grade Entered
57847,57847,Registered,MDJ,25946,KC,NaN,C,MDJ,VCD,Spring 2026,GR,MDJ 25946,CCI,CCI,False,False,False,NaN,2.0,No MT Grade Entered


Well that's weird. The GR indicates those enrolled are graduate students. Graduate students are allowed to take undergraduate courses, but they tend to stick around in the upper level courses, not lower level courses. I did not foresee this happening - I should have checked the classes earlier to catch those graduate students.

Since the population of affected students is small, I am going to do a quick patch fix for them and overwrite the values with the "Passing MT Grades" value.

In [167]:
#Making the Intervention Column Pt. 7 (The Quick Fix)
mthubonly.loc[(mthubonly["MT Grade Status"] == "No MT Grade Entered") & (mthubonly["Mid Term Grade"].notna()),"MT Grade Status"] = "Passing MT Grade" 
mthubonly.groupby("MT Grade Status").count()["Record ID"]

MT Grade Status
Dropped Course                 927
FR/SO Intervention Needed     5460
JR/SR Intervention Needed      652
Never Attended Course           82
No MT Grade Entered           5692
Passing MT Grade             45026
Withdrew From Course           948
Name: Record ID, dtype: int64

**Creating the Cleaned File**

Now that the file has everything we need, we can extract it to a CSV file. This file will be handy to show the population of classes that have higher concentrations of interventions needed or student withdrawals, as well as instances where faculty are not entering midterm grades.

In [168]:
#Creating the cleaned file
mthubonly.to_csv("mthubonlyclean.csv")

**Making the Intervention File**

The next file we want to make is the file with the interventions only. This is what will be used to create the tables that are shown to leadership, as well as the files that advisors would reference when performing their outreach to students.

We only need students who need interventions. As such, we are going to make a version of the file with just the students marked as such. Students who got a passing grade or left the course are not included - the logic is that students are passing/no longer enrolled, so there is no need to contact them.

In [169]:
#Making the Intervention File Pt. 1 (Removing Non-Interventions)
mthubinterventions = mthubonly[((mthubonly["MT Grade Status"] == "FR/SO Intervention Needed") | (mthubonly["MT Grade Status"] == "JR/SR Intervention Needed"))].copy()
mthubinterventions

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Withdrew No MT,Dropped no MT,Dropped no Fin,Final Grade Number,Mid Term Grade Number,MT Grade Status
11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,NaN,2.0,FR/SO Intervention Needed
14,14,Registered,AED,22860,KC,C,D,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,2.0,1.0,FR/SO Intervention Needed
19,19,Registered,AED,22860,KC,B-,C,ARCH,ID,Fall 2022,SO,AED 22860,CAED,CAED,False,False,False,2.7,2.0,FR/SO Intervention Needed
31,31,Stopped Attending - Failed,AED,22860,KC,W,D,ARCH,ID,Fall 2022,SO,AED 22860,CAED,CAED,False,False,False,0.0,1.0,FR/SO Intervention Needed
33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,Fall 2022,FR,AED 22860,CAED,CAED,False,False,False,2.7,1.7,FR/SO Intervention Needed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85275,85275,Registered,MDJ,20288,KC,F,F,MDJ,FM,Fall 2022,SR,MDJ 20288,CotA,CCI,False,False,False,0.0,0.0,JR/SR Intervention Needed
85309,85309,Registered,MDJ,20288,KC,C-,F,MDJ,DMP,Spring 2023,JR,MDJ 20288,CCI,CCI,False,False,False,1.7,0.0,JR/SR Intervention Needed
85315,85315,Registered,MDJ,20288,KC,A-,D,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI,False,False,False,3.7,1.0,FR/SO Intervention Needed
85316,85316,Registered,MDJ,20288,KC,C,C-,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,CCI,False,False,False,2.0,1.7,FR/SO Intervention Needed


A note on this part - at this point I realized that, when I shuffled the data, it mixed in values from the current term across the terms that already had existing data. This made it seem like the prior terms had instances where the instructor never entered a final grade when one should already exist. I had to go back to my dummy file and do some tweaking with it so the non-current term data would be shuffled amongst itself, while the current term data would just stay shuffled alone.

In [170]:
#Making the Intervention File Pt. 2 (Making the CSV file)
mthubinterventions.to_csv("mthubinterventions.csv")